# ⚙️ What is an Optimizer? Why LLMs Use AdamW

Welcome! In this notebook, we answer the fundamental question: <b>What is an optimizer, what are the options, and why does virtually every LLM use AdamW?</b>

---

## 1. The Metaphor: Navigating a Foggy Mountain at Night

Imagine standing on a steep mountain in dense fog at midnight.<br>
Your goal is to reach the <b>lowest valley (minimum loss)</b>.<br>
You can only feel the slope right beneath your feet (the gradient).

<div style='text-align: center; margin: 20px 0;'>
  <img src='assets/optimizer_split.png' width='850' style='border-radius: 8px; border: 1px solid #30363d;' alt='Optimizer and Parameter Split Diagram'>
</div>

---
## 2. Why Do LLMs Break Simple Optimizers (SGD)?

In language and code, words follow <b>Zipf's Law</b>:
<ul>
  <li>Common tokens (spaces, <code>def</code>, <code>import</code>) appear millions of times. Their gradients are <b>massive (e.g. 100.0)</b>.</li>
  <li>Rare tokens (specialized variables, rare syntax) appear once in 10,000 steps. Their gradients are <b>tiny (e.g. 0.001)</b>.</li>
</ul>

<b>The Dilemma for SGD:</b><br>
- If learning rate is large enough for rare words to learn &rarr; Common words explode into NaNs and crash!<br>
- If learning rate is small enough to keep common words safe &rarr; Rare words learn nothing!

---
## 3. Exercise 1: How AdamW Auto-Scales Gradients on Paper 📝

AdamW tracks two running averages for every parameter:
<ol>
  <li><b>m_t (Momentum):</b> Direction of recent gradients.</li>
  <li><b>v_t (Variance):</b> Magnitude squared of recent gradients.</li>
</ol>

The step size for any parameter is approximately:
<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
Effective Step ≈ m_t / sqrt(v_t)
</pre>

### Let's Test Two Extreme Parameters on Paper:
1. <b>Common Word Parameter (Huge Gradient = 100.0):</b><br>
   m_t ≈ 100.0, &nbsp; v_t ≈ 100.0² = 10,000<br>
   Step = 100.0 / sqrt(10,000) = 100.0 / 100.0 = <b>1.0</b> &nbsp; (Tamed down safely!)
<br><br>
2. <b>Rare Word Parameter (Tiny Gradient = 0.001):</b><br>
   m_t ≈ 0.001, &nbsp; v_t ≈ 0.001² = 0.000001<br>
   Step = 0.001 / sqrt(0.000001) = 0.001 / 0.001 = <b>1.0</b> &nbsp; (Boosted so it actually learns!)

<br>
<b>The Magic of AdamW:</b> Every parameter in the 100-billion-parameter model progresses at a normalized, stable speed!

---
## 4. The Parameter Splitting Rule: 2D Matrices vs. 1D RMSNorm Scales

Look at the right panel of the diagram above:<br>
- <b>Group 1 (2D Matrices):</b> Weight Decay = 0.1 (ON) to prevent weights from exploding.<br>
- <b>Group 2 (1D RMSNorm Scales):</b> Weight Decay = 0.0 (OFF) to protect the volume knob from dying!

---
## 5. Verify Parameter Splitting with Real Project Optimizer Code

Now let's run our actual <code>configure_optimizers</code> function from <code>src/engine/optim.py</code> on our Transformer model and verify the parameter groups:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.model.transformer import TransformerDecoder, ModelConfig
from src.engine.optim import configure_optimizers

# Instantiate micro model
config = ModelConfig(vocab_size=1000, dim=128, n_layers=2, n_heads=4, n_kv_heads=2)
model = TransformerDecoder(config)

# Configure optimizer with 0.1 weight decay
optimizer = configure_optimizers(model, weight_decay=0.1, learning_rate=1e-3, betas=(0.9, 0.95))

print(f"\nOptimizer Groups Created: {len(optimizer.param_groups)}")
for i, group in enumerate(optimizer.param_groups):
    sample_param = group['params'][0]
    print(f"Group {i+1}: Weight Decay = {group['weight_decay']:<4} | Parameter Count = {len(group['params']):<3} | Sample Shape = {list(sample_param.shape)}")

# Verify that Group 2 has strictly weight_decay == 0.0
assert optimizer.param_groups[1]['weight_decay'] == 0.0
print("\nSUCCESS: RMSNorm and 1D parameters are 100% protected with weight_decay = 0.0!")